# NB6 · Web arayüzü

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapılıyor

Beş defterde yazdığınız program çalışıyor ama yalnızca Colab içinden kullanılabiliyor.
Bu defterde ona bir web arayüzü ekleyeceksiniz. Arayüz tarayıcıdan açılan bir bağlantı
üretir; klinisyene gösterebileceğiniz hâle gelir.

Ekranda tek bir sayı göstermeyeceksiniz. Kararın kendisi, bariyerlerin durumu ve gerekçe
birlikte görünecek. 16 Eylül dersinde anlatılan ayrım buydu: Karar destek sisteminin
ürünü bir teşhis değil, gerekçesiyle sunulan bir dikkat yönlendirmesidir.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')

!pip -q install gradio


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda toplanan bloğun tamamını aşağıdaki hücreye yapıştırınız.
İlk satırdaki `#@cdss` işaretini silmeyiniz; o blok bu defterin sonunda yeniden
toplanacak ve bir sonrakine taşınacaktır.

Blok çalıştığında önceki defterlerde yazdığınız her şey yeniden kurulur. İnternetten
veri okuyan satırlar varsa bu hücre birkaç saniye sürebilir.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol · Gelen kod


In [ ]:
kit.check_defined('model', 'guvenli_tahmin', 'hasta_gerekcesi', 'uyumluluk_raporu',
                  'gerekce_tablosu', 'sonuc')


---

## Adım 1 · Arayüzün giriş alanları

Modelin kullandığı bilgi sayısı otuzu geçebilir. Her biri için ekranda bir alan koymak
arayüzü kullanılamaz hâle getirir.

Bu yüzden en önemli altı bilgi seçilir, kalanlar eğitim grubunun tipik değerlerinde
tutulur. Seçim NB4'te ürettiğiniz `gerekce_tablosu` tablosundan yapılır.

Bu seçimin bir bedeli vardır ve ekranda yazması gerekir: Kullanıcı, göremediği
bilgilerin varsayılan değerde tutulduğunu bilmelidir.


### İstem 1

```
Arayüzde gösterilecek bilgileri seçen bir kod yaz.

gerekce_tablosu adlı tabloda bilgiler etkilerine göre sıralı duruyor. En etkili altı
sayısal bilgiyi seç.

Ayrıca her bilgi için eğitim grubundaki tipik değeri, en düşük ve en yüksek makul değeri
hesapla; arayüzdeki kaydırıcıların sınırları bunlar olacak. Uç noktalarda tek tük
görülen değerler sınırları bozmasın.

Sonuçları su adlarla sakla:
  ARAYUZ_BILGILERI -> gösterilecek bilgilerin adları, liste olarak
  VARSAYILANLAR    -> her bilginin tipik değeri
  SINIRLAR         -> her bilgi için en düşük ve en yüksek değer

Ekrana hangi bilgilerin seçildiğini ve kaç bilginin gizli kaldığını yaz.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
ARAYUZ_BILGILERI, VARSAYILANLAR ve SINIRLAR adında üç nesne hazır olmalı.
ARAYUZ_BILGILERI altı adet bilgi adı içermeli.
```


In [ ]:
#@cdss arayuz_alanlari
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_defined('ARAYUZ_BILGILERI', 'VARSAYILANLAR', 'SINIRLAR')
print('\nSeçilen bilgiler:', list(ARAYUZ_BILGILERI))


---

## Adım 2 · Arayüz

Şimdi arayüzün kendisini kurdurcaksınız.

Bir noktaya dikkat ediniz. Cinsiyet, sigorta ve benzeri demografik bilgiler gerekçe
listesinde görünebilir. Bunlar klinisyene sunulacak bir sebep değildir; ayrı bir uyarı
satırına taşınmalıdır. Bir modelin demografik bir bilgiden öğrenmiş olması bir adalet
denetimi bulgusudur ve sistemin devreye alınmasını engelleyebilir.


### İstem 2

```
Gradio ile tek sayfalık bir web arayüzü kur.

Elimde şunlar var:
  guvenli_tahmin    -> bir hastanın bilgilerini alır; karar, olasilik ve gerekce
                       anahtarlarını içeren bir sözlük döndürür
  hasta_gerekcesi   -> bir hastanın bilgilerini alır; bilgi, deger ve katki sütunlu
                       bir tablo döndürür
  ARAYUZ_BILGILERI  -> ekranda gösterilecek bilgilerin adları
  VARSAYILANLAR     -> her bilginin tipik değeri
  SINIRLAR          -> kaydırıcıların alt ve üst sınırları

Arayüz şöyle olsun:
1. Üstte başlık ve şu uyarı: Bu bir öğretim prototipidir, doğrulanmış bir klinik araç
   değildir ve gerçek hasta kararlarında kullanılamaz.
2. Solda her bilgi için bir kaydırıcı ve bir Değerlendir düğmesi.
3. Sağda kararın metin olarak gösterimi ve gerekçe tablosu.
4. Ekranda kaç bilginin gösterilmediğini ve bunların varsayılan değerde tutulduğunu
   belirt.
5. Gerekçe tablosunda cinsiyet, sigorta, medeni hâl gibi demografik bir bilgi varsa onu
   tablodan çıkar ve yerine ayrı bir adalet denetimi uyarısı göster.
6. Sistem tahmin üretmediğinde, yani bariyerlerden birine takıldığında, gerekçesini
   açıkça yaz.

Değerlendirme işini yapan kısmı degerlendir adında ayrı bir işlem parçası olarak yaz;
arayüz o parçayı çağırsın. Böylece arayüzü açmadan da sınayabiliriz.

Son satırda arayüzü paylaşılabilir bağlantıyla başlat.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
degerlendir adında çalıştırılabilir bir işlem parçası olmalı; kaydırıcı değerlerini
sırayla alsın ve iki şey döndürsün: Bir metin ve bir tablo.
arayuz adında bir Gradio nesnesi hazır olmalı.
```


In [ ]:
#@cdss arayuz
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2

Aşağıdaki hücre arayüzü açmadan değerlendirme işlem parçasını doğrudan sınar.


In [ ]:
kit.check_function('degerlendir')


In [ ]:
girdiler = [float(VARSAYILANLAR[b]) for b in ARAYUZ_BILGILERI]
metin, tablo = degerlendir(*girdiler)
print('OLAĞAN HASTA'); print(metin)
print()
uc = [float(SINIRLAR[b][1]) * 5 for b in ARAYUZ_BILGILERI]
metin_uc, _ = degerlendir(*uc)
print('UÇ DEĞERLİ HASTA'); print(metin_uc.split(chr(10))[0])


### Python notu · Geri çağrı

Arayüz kodunda `dugme.click(degerlendir, ...)` gibi bir satır göreceksiniz. Burada
`degerlendir` çağrılmıyor; arayüze **veriliyor**.

Buna **geri çağrı** denir. Arayüz düğmeye basıldığında o işlem parçasını kendisi
çalıştırır. Sizin göreviniz işi tanımlamak, ne zaman çalışacağına arayüz karar veriyor.

Bu ayrım pratikte işinize yarar: Değerlendirme işini ayrı bir parça olarak yazdırdığımız
için, yukarıdaki hücrede arayüzü hiç açmadan sınayabildik. Arayüzle iş mantığını ayrı
tutmak, yazılım geliştirmede yerleşik bir alışkanlıktır.


---

## Adım 3 · Arayüzü açmak


In [ ]:
arayuz.launch(share=True)


### Arayüzde denenecekler

Beş deneme yapınız.

Varsayılan değerlerle çalıştırınız ve kararın ne olduğuna bakınız.

Kaydırıcıları yavaşça hareket ettirerek olasılığı eşiğe yaklaştırınız. Sistemin kararsız
kalmaya başladığı noktayı bulunuz. Klinik uygulamada bu bant, kimin listeye alınacağını
belirler.

Birkaç kaydırıcıyı uçlara çekiniz. Sistem tahmin üretmeyi bırakmalıdır.

Farklı kaydırıcı birleşimleriyle benzer bir olasılığa ulaşınız ve gerekçe tablosunun
değiştiğini görünüz. İki hasta aynı riski taşıyabilir ama klinisyenin yapması gereken
farklıdır.

Son olarak, eğer arayüzünüzde varsa, demografik bir bilgiyi değiştirip başka hiçbir şeye
dokunmayınız. Olasılık değişiyorsa ve adalet uyarısı çıkıyorsa model o bilgiden öğrenmiş
demektir.


---

## Programın tamamı

Aşağıdaki hücre altı defterde yazdığınız kodun tamamını toplar ve tek bir dosya olarak
kaydeder. Bu dosya sizin klinik karar destek sisteminizdir.

Colab oturumu kapandığında dosya silinir. Sol taraftaki dosya panelinden indiriniz.


In [ ]:
program = kit.export(save_as='cdss_sistem.py')

print()
print('Adım sayısı :', len(kit.steps()))
print('Satır sayısı:', program.count(chr(10)))


## Atölyenin sonu

Altı defterde bir klinik karar destek sistemi yazdınız. Sistem veriye ulaşıyor, veriyi
hazırlıyor, model kuruyor, başarımını ölçüyor, kararını gerekçelendiriyor, güvenlik
bariyerleri uyguluyor, uyumluluk raporu üretiyor ve web üzerinden çalışıyor.

Bunların hiçbirini elle kodlamadınız. Her adımda ne istediğinizi tarif ettiniz, gelen
kodu sınadınız ve gerektiğinde düzelttirdiniz.

Prototipin çalışıyor olması hazır olduğu anlamına gelmez. Eksik olanlar şunlardır: Başka
bir merkezde dış doğrulama, klinisyen kaynaklı fizyolojik sınırlar, iş akışına yerleştirme
çalışması, eşiklerin klinik ekipçe onaylanması, düzenleyici sınıflandırma, kişisel veri
değerlendirmesi ve devreye alma sonrası başarım izlemesi.

Bu liste, iki saatte yapılan işin klinik bir ürün yolculuğunun neresinde durduğunu
gösterir. Kod üretme maliyeti düşmüştür; geriye kalan hiçbir şey düşmemiştir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelir ve Türkiye'deki bir yoğun
bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
